: 

In [1]:
import ollama

client = ollama.Client()
[m["model"] for m in client.list()["models"]]

['qwen3:8b', 'qwen3.6:latest']

In [8]:
response = client.chat(
    model="qwen3:8b",
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
    stream = True
)


In [11]:
import ollama

client = ollama.Client()

def run_agent(model: str, user_input: str, tools=None, max_turns: int = 10):
    messages = [{"role": "user", "content": user_input}]
    tools_by_name = {t.__name__: t for t in (tools or [])}

    for _ in range(max_turns):
        # 1. TURN: send the whole conversation so far, get one response back
        response = client.chat(
            model=model,
            messages=messages,
            tools=tools or None,
        )
        message = response["message"]
        messages.append(message)  # remember what the model said

        # 2. Did the model ask to call a tool, or is it done talking?
        tool_calls = message.get("tool_calls")
        if not tool_calls:
            return message  # final answer, loop ends

        # 3. Run each requested tool, feed the result back as a new message
        for call in tool_calls:
            name = call["function"]["name"]
            args = call["function"]["arguments"]
            fn = tools_by_name.get(name)
            try:
                result = fn(**args) if fn else f"Error: unknown tool '{name}'"
            except Exception as e:
                result = f"Error running '{name}': {e}"

            messages.append({
                "role": "tool",
                "content": str(result),
                "tool_name": name,
            })
        # loop back to step 1 — model now sees the tool result and continues


# example
answer = run_agent("qwen3:8b", "Say hello in one sentence.")
print(answer) 

role='assistant' content='Hello! How can I assist you today?' thinking='Okay, the user wants me to say hello in one sentence. Let me think about how to approach this. First, I need to make sure the response is friendly and concise. Since it\'s just a single sentence, I should avoid any unnecessary words.\n\nMaybe start with a simple greeting like "Hello!" to keep it direct. Then add a welcoming phrase to make it more personable. Something like "How can I assist you today?" would be good because it invites the user to ask for help, which is appropriate for a conversational AI.\n\nWait, should I make it more casual? Maybe "Hi there!" instead of "Hello!"? But "Hello" is more formal and might be better for a general audience. Also, the user might prefer a friendly tone, so perhaps adding an emoji could make it more approachable. However, the user didn\'t specify if they want an emoji, so maybe it\'s safer to keep it without. \n\nLet me check if the sentence is clear and to the point. "Hell

In [ ]:
answer